# Vaidya — Phase 4: DPO Fine-tuning + Adapter Merge

**Steps:**
1. Generate 500 DPO preference pairs from val set
2. DPO train on top of Phase 2 SFT checkpoint (run3)
3. Merge DPO adapter into base model
4. Push merged model to HuggingFace Hub + S3

**GPU:** A10G or L4 (24GB) — ~30–45 min DPO training on 500 pairs

## Step 1 — Install packages

In [ ]:
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    trl==0.15.2 \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3 \
    sentencepiece \
    huggingface_hub \
    "fsspec[http]==2024.9.0" \
    "packaging==24.2"

!pip install -q bitsandbytes --upgrade --force-reinstall

print('✅ Install done')
print('⚠️  RESTART THE KERNEL, then run from Step 2.')

## Step 2 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available()
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CC   : {torch.cuda.get_device_capability()}')

## Step 3 — Set working directory

In [ ]:
import os
work_dir = '/teamspace/studios/this_studio/vaidya/layer3_advanced'
os.chdir(work_dir)
print(f'Working dir: {os.getcwd()}')

## Step 4 — AWS credentials

In [ ]:
import os, boto3

# ── Fill in your credentials ────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
# ────────────────────────────────────────────────────────────

s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
assert 'vaidya-artifacts' in buckets
print('✅ S3 connected')

## Step 5 — Generate DPO pairs

In [ ]:
# Generates 500 (prompt, chosen, rejected) pairs from val set → dpo_pairs.jsonl → S3
!python generate_dpo_pairs.py --n-pairs 500

## Step 6 — DPO training

Trains on top of Phase 2 SFT checkpoint (run3). ~30–45 min on L4/A10G.

In [ ]:
!python dpo_train.py --sft-run-name run3 --run-name dpo-run1

## Step 7 — HuggingFace login (for Hub push)

In [ ]:
from huggingface_hub import login
# Get token from https://huggingface.co/settings/tokens
login(token='YOUR_HF_TOKEN')

## Step 8 — Merge adapters + push to Hub + S3

Merges SFT + DPO adapters into base model. ~10–15 min.

In [ ]:
!python merge_adapter.py --sft-run-name run3 --dpo-run-name dpo-run1

## Gate check — before Phase 5

- ✅ `dpo_pairs.jsonl` at `s3://vaidya-artifacts/data/`
- ✅ DPO checkpoint at `s3://vaidya-artifacts/checkpoints/dpo-run1/`
- ✅ Merged model at `s3://vaidya-artifacts/models/merged/`
- ✅ Model live at `https://huggingface.co/Anand09-in/vaidya-mistral-7b-medmcqa`
- ✅ Latency benchmark logged (p50 / p95 ms)

**Next:** Phase 5 — Quantization (GPTQ / AWQ / GGUF) + benchmark table